# Daily Scanner Candidates Replay View v0.1

Research-only notebook for inspecting `daily_scanner_candidates_table_v0_1` controlled replays.

Authority remains in Data Foundation contracts, registry, validators, manifests and the builder script. This notebook does not define scanner semantics and must not be used as an official materializer.

## Canonical References

- Builder: `01_TSIS_DATA_FOUNDATION/scripts/materialize_daily_scanner_candidates_table.py`
- Target contract: `01_foundations/module_contracts/outputs/daily_scanner_candidates_table_target_contract_v0_1.md`
- Scanner framework: `01_foundations/module_contracts/outputs/scanner_framework_and_definitions_contract_v0_1.md`
- Scanner configs: `configs/data_foundation_outputs/scanner_definitions/`
- Controlled replay evidence root: `C:/TSIS_Data/tests/test_runs/2026-06-29/daily_scanner_candidates_replay_20250102_20250110_v0_1/`

The scanner tells us where to look. It is not a final market state, label, reward, strategy signal, execution truth or live authority.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import duckdb
import pandas as pd

REPO_ROOT = Path(r"C:/TSIS_Data")
MODULE_ROOT = REPO_ROOT / "01_TSIS_DATA_FOUNDATION"
BUILDER = MODULE_ROOT / "scripts" / "materialize_daily_scanner_candidates_table.py"

DEFAULT_REPLAY_ROOT = REPO_ROOT / "tests" / "test_runs" / "2026-06-29" / "daily_scanner_candidates_replay_20250102_20250110_v0_1"
DEFAULT_MANIFEST = DEFAULT_REPLAY_ROOT / "_daily_scanner_candidates_table_manifest_v0_1_candidate_replay.json"
DEFAULT_DATASET = DEFAULT_REPLAY_ROOT / "daily_scanner_candidates_table_v0_1_candidate_replay" / "data.parquet"
DEFAULT_SUMMARY = DEFAULT_REPLAY_ROOT / "_daily_scanner_candidates_table_summary_v0_1_candidate_replay.csv"

print("Builder:", BUILDER)
print("Replay root:", DEFAULT_REPLAY_ROOT)
print("Manifest exists:", DEFAULT_MANIFEST.exists())
print("Dataset exists:", DEFAULT_DATASET.exists())

## Optional: Run A New Controlled Replay

Keep ranges small while inspecting. The output goes to `C:/TSIS_Data/tests/test_runs/...`, not to the official E-root.

In [ ]:
RUN_NEW_REPLAY = False

START_DATE = "2025-01-02"
END_DATE = "2025-01-10"
RUN_ID = "daily_scanner_candidates_replay_notebook_20250102_20250110_v0_1"
OUTPUT_ROOT = REPO_ROOT / "tests" / "test_runs" / "2026-06-29" / RUN_ID

if RUN_NEW_REPLAY:
    cmd = [
        sys.executable,
        str(BUILDER),
        "--start-date", START_DATE,
        "--end-date", END_DATE,
        "--run-id", RUN_ID,
        "--output-root", str(OUTPUT_ROOT),
        "--overwrite",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    REPLAY_ROOT = OUTPUT_ROOT
else:
    REPLAY_ROOT = DEFAULT_REPLAY_ROOT

MANIFEST_PATH = REPLAY_ROOT / "_daily_scanner_candidates_table_manifest_v0_1_candidate_replay.json"
DATASET_PATH = REPLAY_ROOT / "daily_scanner_candidates_table_v0_1_candidate_replay" / "data.parquet"
SUMMARY_PATH = REPLAY_ROOT / "_daily_scanner_candidates_table_summary_v0_1_candidate_replay.csv"

print("Active replay root:", REPLAY_ROOT)
print("Manifest:", MANIFEST_PATH)
print("Dataset:", DATASET_PATH)
assert MANIFEST_PATH.exists(), MANIFEST_PATH
assert DATASET_PATH.exists(), DATASET_PATH

In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
summary = pd.read_csv(SUMMARY_PATH)

print("status:", manifest["status"])
print("promotion_level:", manifest["promotion_level"])
print("run_id:", manifest["scanner_run_id"])
print("scope:")
display(pd.Series(manifest["replay_scope"]))
print("validations:")
display(pd.Series(manifest["validations"]))
print("summary:")
display(summary)

In [ ]:
con = duckdb.connect()
parquet = DATASET_PATH.as_posix()

columns = con.sql(f"describe select * from read_parquet('{parquet}')").fetchdf()
print("column_count:", len(columns))
display(columns.head(80))

## Scanner Comparison

`trade_station_like_scanner_v0_1` reproduces the narrow operational hot-list idea. `broad_in_play_discovery_scanner_v0_1` is designed to protect research from late-arrival bias.

In [ ]:
comparison = con.sql(f"""
select
    session_date,
    scanner_definition_id,
    count(*) as evaluated_rows,
    sum(case when selected_trade_station_like_top25 then 1 else 0 end) as trade_station_like_top25_rows,
    sum(case when selected_broad_discovery then 1 else 0 end) as broad_discovery_rows,
    sum(case when selected_broad_discovery and volume_today < 500000 then 1 else 0 end) as broad_below_500k_rows,
    min(population_denominator_count) as denominator_min,
    max(population_denominator_count) as denominator_max
from read_parquet('{parquet}')
group by 1, 2
order by 1, 2
""").fetchdf()
display(comparison)

In [ ]:
trade_station_like = con.sql(f"""
select
    session_date,
    rank,
    ticker,
    last_price,
    pct_chg_1d,
    volume_today,
    market_cap_usd,
    candidate_reasons
from read_parquet('{parquet}')
where selected_trade_station_like_top25
order by session_date, rank
limit 50
""").fetchdf()
display(trade_station_like)

In [ ]:
broad_below_500k = con.sql(f"""
select
    session_date,
    ticker,
    last_price,
    pct_chg_1d,
    gap_pct,
    volume_today,
    rvol_to_time,
    rank_composite_in_play,
    candidate_reasons
from read_parquet('{parquet}')
where selected_broad_discovery
  and volume_today < 500000
order by session_date, rank_composite_in_play nulls last, pct_chg_1d desc nulls last
limit 75
""").fetchdf()
display(broad_below_500k)

## Prohibition Checks

These must stay zero for this controlled replay. Scanner rows are not ML/RL/live authority.

In [ ]:
prohibition = con.sql(f"""
select
    sum(case when full_universe_claim then 1 else 0 end) as full_universe_claim_true_rows,
    sum(case when valid_for_ml_feature_candidate then 1 else 0 end) as ml_feature_candidate_rows,
    sum(case when valid_for_rl_state_candidate then 1 else 0 end) as rl_state_candidate_rows,
    sum(case when valid_for_live_downstream_candidate then 1 else 0 end) as live_downstream_candidate_rows,
    sum(case when contains_future_information_without_event_filter then 1 else 0 end) as future_information_flagged_rows
from read_parquet('{parquet}')
""").fetchdf()
display(prohibition)

In [ ]:
dedup = manifest.get("deduplication_policies", {})
print("Deduplication policies from manifest:")
display(pd.Series(dedup))